# Food Waste Estimation -- Inference Demo

Load a trained checkpoint and predict leftover weight from a before/after image pair.
Supports single-model and ensemble (all fold checkpoints averaged) inference.

**Run cells top to bottom. Do not skip the setup cell.**

## 0. Environment Setup

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ml-food-waste-estimation'  # adjust if needed
    os.chdir(PROJECT_ROOT)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    PROJECT_ROOT = os.getcwd()

sys.path.insert(0, 'src')
print(f'{"Colab" if IN_COLAB else "Local"} | cwd: {os.getcwd()}')

## 1. Install Dependencies

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Dependencies installed.')
else:
    print('Local: run "uv sync" in terminal if not done yet.')

## 2. Imports

In [ ]:
%matplotlib inline
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from dataset import get_transforms
from model import DualStreamEfficientNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 3. Specify Image Paths

Set `RAW_INPUT = False` if your images are already segmented (black background, white plate).  
Set `RAW_INPUT = True` if your images are raw photos -- Section 3b will auto-segment them.

**Local**: edit the paths below.  
**Colab**: the upload widget will appear automatically.

In [ ]:
RAW_INPUT = False  # set True if providing raw (unsegmented) photos

if IN_COLAB:
    from google.colab import files as _files
    print('Upload the BEFORE image:')
    _up = _files.upload()
    BEFORE_PATH = list(_up.keys())[0]
    print('Upload the AFTER image:')
    _up = _files.upload()
    AFTER_PATH = list(_up.keys())[0]
else:
    if RAW_INPUT:
        # Point to raw photos -- Section 3b will segment them automatically
        BEFORE_PATH = 'data/raw/data_before/001/001_001_DSC_0059_bef.JPG'
        AFTER_PATH  = 'data/raw/data_after/001/001_001_DSC_0108_aft.JPG'
    else:
        # Pre-segmented images (default)
        BEFORE_PATH = 'data/segmented/data_before/001_001_001_DSC_0059_bef.JPG'
        AFTER_PATH  = 'data/segmented/data_after/001_001_001_DSC_0108_aft.JPG'

# Optional: known serving weight in grams for gram-level output. Set to None to skip.
WEIGHT_BEFORE_G = None  # e.g. 250.0

print(f'Before      : {BEFORE_PATH}')
print(f'After       : {AFTER_PATH}')
print(f'Raw input   : {RAW_INPUT}')
print(f'Serving wt  : {WEIGHT_BEFORE_G}g' if WEIGHT_BEFORE_G else 'Serving wt  : not provided')
if not os.path.exists(BEFORE_PATH):
    print('WARNING: before image not found.')
if not os.path.exists(AFTER_PATH):
    print('WARNING: after image not found.')

## 3b. Auto-Segmentation (only runs when `RAW_INPUT = True`)

Converts raw food photos into the ground-truth format:
- **Background** (checkered mat) -> black (0, 0, 0)
- **Plate surface** (including rim) -> pure white (255, 255, 255)
- **Food** -> original colors preserved

Skip this section if `RAW_INPUT = False`.

In [ ]:
if RAW_INPUT:
    import tempfile
    from segmentation import segment_image

    _seg_dir = tempfile.mkdtemp()
    _before_seg_path = os.path.join(_seg_dir, 'before_seg.jpg')
    _after_seg_path  = os.path.join(_seg_dir, 'after_seg.jpg')

    print('Segmenting before image ...')
    before_seg_img = segment_image(BEFORE_PATH, _before_seg_path)
    print('Segmenting after image ...')
    after_seg_img  = segment_image(AFTER_PATH,  _after_seg_path)

    # Show raw vs segmented side-by-side for both images
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    axes[0, 0].imshow(Image.open(BEFORE_PATH))
    axes[0, 0].set_title('Before -- raw')
    axes[0, 0].axis('off')
    axes[0, 1].imshow(before_seg_img)
    axes[0, 1].set_title('Before -- segmented (800x800)')
    axes[0, 1].axis('off')
    axes[1, 0].imshow(Image.open(AFTER_PATH))
    axes[1, 0].set_title('After -- raw')
    axes[1, 0].axis('off')
    axes[1, 1].imshow(after_seg_img)
    axes[1, 1].set_title('After -- segmented (800x800)')
    axes[1, 1].axis('off')
    plt.suptitle('Auto-segmentation: raw -> ground-truth format', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Update paths so the rest of the notebook uses the segmented versions
    BEFORE_PATH = _before_seg_path
    AFTER_PATH  = _after_seg_path
    print(f'Using segmented images for inference: {_seg_dir}')
else:
    print('RAW_INPUT = False -- skipping segmentation, using images as-is.')

## 4. Load Checkpoint

In [ ]:
CHECKPOINT_DIR = 'checkpoints'

checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'fold_*_best.pth')))
print(f'Found {len(checkpoints)} checkpoint(s):')
for c in checkpoints:
    print(f'  {c}')

if not checkpoints:
    raise FileNotFoundError(
        'No checkpoints found. Run training.ipynb first.'
    )

## 5. Predict

**Single model**: uses fold 1 checkpoint.  
**Ensemble**: averages predictions across all available fold checkpoints.

In [ ]:
transform = get_transforms('val')


def load_model(checkpoint_path):
    ckpt  = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model = DualStreamEfficientNet(pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    return model


def predict_single(model, before_t, after_t, area_ratio_t):
    with torch.no_grad():
        return float(model(before_t, after_t, area_ratio_t).item())


# Prepare inputs once -- reused by single-model and ensemble
_before_area = float(np.any(np.array(Image.open(BEFORE_PATH).convert('RGB')) > 0, axis=2).sum())
_after_area  = float(np.any(np.array(Image.open(AFTER_PATH).convert('RGB')) > 0, axis=2).sum())
_ar          = min(_after_area / _before_area, 1.0) if _before_area > 0 else 0.0

before_t     = transform(Image.open(BEFORE_PATH).convert('RGB')).unsqueeze(0).to(DEVICE)
after_t      = transform(Image.open(AFTER_PATH).convert('RGB')).unsqueeze(0).to(DEVICE)
area_ratio_t = torch.tensor([[_ar]], dtype=torch.float32).to(DEVICE)

# Single-model inference (fold 1)
model    = load_model(checkpoints[0])
r_single = predict_single(model, before_t, after_t, area_ratio_t)

print('=== Single Model (Fold 1) ===')
print(f'  Consumption ratio : {r_single:.4f}')
print(f'  Area ratio        : {_ar:.4f}')
if WEIGHT_BEFORE_G:
    print(f'  Weight after      : {r_single * WEIGHT_BEFORE_G:.1f}g')
    print(f'  Leftover          : {(1 - r_single) * WEIGHT_BEFORE_G:.1f}g')

# Final display values -- overwritten by ensemble when all folds available
disp_ratio = r_single
disp_label = 'Single model (fold 1)'

# Ensemble inference (all folds)
if len(checkpoints) > 1:
    all_ratios = []
    for ckpt_path in checkpoints:
        m = load_model(ckpt_path)
        all_ratios.append(predict_single(m, before_t, after_t, area_ratio_t))
        del m
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    disp_ratio = float(np.mean(all_ratios))
    disp_label = f'Ensemble ({len(checkpoints)} folds)'

    print(f'\n=== Ensemble ({len(checkpoints)} folds) ===')
    print(f'  Consumption ratio : {disp_ratio:.4f} +/- {np.std(all_ratios):.4f}')
    if WEIGHT_BEFORE_G:
        print(f'  Weight after      : {disp_ratio * WEIGHT_BEFORE_G:.1f}g')
        print(f'  Leftover          : {(1 - disp_ratio) * WEIGHT_BEFORE_G:.1f}g')

## 6. Visualize

In [ ]:
before_img = Image.open(BEFORE_PATH)
after_img  = Image.open(AFTER_PATH)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(before_img)
axes[0].set_title('Before')
axes[0].axis('off')

after_title = f'After\nConsumption ratio: {disp_ratio:.3f}'
if WEIGHT_BEFORE_G:
    after_title += f'\nWeight after: {disp_ratio * WEIGHT_BEFORE_G:.1f}g'
axes[1].imshow(after_img)
axes[1].set_title(after_title)
axes[1].axis('off')

plt.suptitle(disp_label, fontsize=12)
plt.tight_layout()
plt.show()